# 第12周 · 训练小型语言模型（Capstone 实验设计自审）

**本周结束时你能做到**：解释自回归语言建模——"预测下一个字符"就是一个分类问题，损失是交叉熵（平均负对数似然，NLL）；从计数 bigram 出发，用梯度下降训练出等价的神经 bigram；照着骨架搭出字符级 GPT，在 Tiny Shakespeare 上训练并用温度采样生成文本；用困惑度、长度分布和新颖度评估一个生成模型；理解 **vectoring**，并完成 Capstone 实验设计的自审。

本周三份蓝本，都来自 Andrej Karpathy：

- S1 照 **makemore 第 1 讲**（视频 [The spelled-out intro to language modeling: building makemore](https://www.youtube.com/watch?v=PaCmpygFfXo)；仓库 [karpathy/makemore](https://github.com/karpathy/makemore)，MIT 许可；`names.txt` 是美国社会保障局 2018 年最常见的 32,033 个婴儿名）；
- S2 的模型骨架照 **"Let's build GPT"** 的 `gpt.py`（视频 [Let's build GPT: from scratch, in code, spelled out](https://www.youtube.com/watch?v=kCc8FmEb1nY)；仓库 [karpathy/ng-video-lecture](https://github.com/karpathy/ng-video-lecture)，MIT 许可）；
- 延伸阅读：**microgpt**（见 S1 末尾）。

继续上周的规矩：每行张量运算旁边写 shape 注释。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import math
import random
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from zero2researcher import data, experiments
from zero2researcher.checks import week12 as ck

# 课程统一的 device 选择：有 CUDA 用 CUDA（Colab GPU），Apple 芯片用 MPS，否则 CPU
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

---
## S1 · 语言模型从计数开始：bigram（makemore 第 1 讲）

**语言模型** = 给"下一个字符是什么"打分的模型。最简单的版本只看**前一个**字符：**bigram** 模型 $P(\text{下一个} \mid \text{当前})$。

数据是 32,033 个英文名字。注意加载函数是 `data.load_baby_names()`——别和第 1 周成绩单上 30 个学生名的 `load_names()` 搞混：

In [ ]:
words = data.load_baby_names()
print(len(words), "个名字，例如", words[:8])
print("最短", min(len(w) for w in words), "个字母，最长", max(len(w) for w in words), "个字母")

一个名字里藏着好几个 bigram。用特殊字符 `.` 标记名字的**开始**和**结束**（模型还要学会"什么时候该停"）：`emma` → `(., e) (e, m) (m, m) (m, a) (a, .)`。字符表 = 26 个字母 + `.` = 27 个，`.` 编号为 0：

In [ ]:
chars = sorted(set("".join(words)))        # 26 个小写字母
stoi = {ch: i + 1 for i, ch in enumerate(chars)}
stoi["."] = 0                              # '.' = 名字的开始与结束
itos = {i: ch for ch, i in stoi.items()}
print(len(stoi), "个字符:", "".join(itos[i] for i in range(27)))

### 数一数：27 × 27 的计数矩阵

`N[a, b]` = 字符 a 后面紧跟字符 b 的次数：

In [ ]:
N = torch.zeros((27, 27), dtype=torch.int32)
# 在这里写你的代码
raise NotImplementedError
print("bigram 总数:", N.sum().item())
top = torch.argsort(N.flatten(), descending=True)[:5]
print("最常见的 5 个:", [(itos[i // 27] + itos[i % 27], N.flatten()[i].item()) for i in top.tolist()])

fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.imshow(N, cmap="Blues")
labels = [itos[i] for i in range(27)]
ax.set_xticks(range(27), labels)
ax.set_yticks(range(27), labels)
ax.set_xlabel("Next character")
ax.set_ylabel("Current character")
ax.set_title("The densest cells are name endings: 'n.' and 'a.'")
plt.show()

### 从计数到概率：+1 平滑与一个经典的广播 bug

每一行除以该行之和，就得到条件概率 $P(b \mid a)$。先给每个格子加 1（**+1 平滑**）：没见过的组合也有一点点概率，否则一个从没出现过的 bigram 会让概率变成 0、对数变成 $-\infty$。

做除法时 `keepdim=True` 是关键——makemore 视频专门讲过这个坑：

In [ ]:
# 在这里写你的代码
raise NotImplementedError
print("正确：前 5 行之和", P.sum(dim=1)[:5].numpy().round(4))

P_bug = (N + 1).float() / (N + 1).float().sum(dim=1)     # (27, 27) / (27,)：广播成按"列"相除！
print("忘了 keepdim：前 5 行之和", P_bug.sum(dim=1)[:5].numpy().round(4), "← 不报错，但已经错了")

`(27,)` 在广播时被当成**行向量** `(1, 27)`，于是每一**列**除以了某一行的和——不报错、不发散，只是悄悄算错。读广播规则时永远从最后一维往前对齐。

更阴险的是：这个 bug 在这份数据上**连 NLL 都看不出来**（下面评估一节会验证）。原因是每个名字都以 `.` 开头、以 `.` 结尾，每个字符作为"前一个"出现的次数恰好等于它作为"后一个"出现的次数，于是"除以前一个字符的行和"和"除以后一个字符的行和"在平均对数似然里一模一样。单一指标不可靠——**要直接检查不变量**：每一行加起来必须是 1。本周的 `ck.bigram_nll` 因此要你把 `P` 也交上来。

### 采样：让模型"写"名字

从 `.` 出发，按 `P` 的那一行抽下一个字符，再以它为当前字符继续抽，直到抽到 `.`：

In [ ]:
g = torch.Generator().manual_seed(2147483647)    # 与 makemore 视频同一个种子
for _ in range(8):
    ix, out = 0, []
    # 在这里写你的代码
    raise NotImplementedError
    print("".join(out))

很烂，但比瞎猜强——bigram 只记得前一个字母。怎么**量化**"比瞎猜强多少"？

### 评估：平均负对数似然（NLL）

好模型应该给真实数据高概率。把训练集里每个 bigram 的概率相乘得到**似然**；乘积太小，取对数变加法；再取负、求平均：

$$\text{NLL} = -\frac{1}{n}\sum_{i=1}^{n} \log P(c_{i+1} \mid c_i)$$

越低越好。这正是第 5、8 周的**交叉熵**：语言建模就是"在 27 个类别里分类下一个字符"。两个参照：均匀瞎猜每个字符概率 $1/27$，NLL = $\ln 27 = 3.296$；**困惑度**（perplexity）$= e^{\text{NLL}}$，可以读成"模型每一步相当于在几个字符里均匀地犹豫"。

In [ ]:
# 在这里写你的代码
raise NotImplementedError
uniform = math.log(27)
print(f"{n:,} 个 bigram 的平均 NLL = {nll:.4f} nats，困惑度 = {math.exp(nll):.1f}")
print(f"均匀瞎猜：NLL = ln 27 = {uniform:.4f} nats，困惑度 = 27")
ck.bigram_nll(nll, P=P)          # 同时检查 P 的每一行之和为 1
ck.uniform_nll(uniform)

In [ ]:
# 回到那个广播 bug：用同一套 NLL 评估 P_bug——两个数只差浮点舍入
xs_all = torch.tensor([stoi[a] for w in words for a in "." + w])     # 每个 bigram 的"前一个"
ys_all = torch.tensor([stoi[b] for w in words for b in w + "."])     # 每个 bigram 的"后一个"
print(f"P 的 NLL {-P[xs_all, ys_all].log().mean().item():.6f} | P_bug 的 NLL {-P_bug[xs_all, ys_all].log().mean().item():.6f}")
print("P_bug 的前 5 行之和:", P_bug.sum(dim=1)[:5].numpy().round(4), "← 只有检查行和才发现得了")
ck.row_normalized(P)

In [ ]:
# 为什么要平滑：一个含有从没出现过的 bigram（'jq'）的名字
P0 = N.float() / N.float().sum(dim=1, keepdim=True)          # 不平滑
for label, PP in [("不平滑", P0), ("+1 平滑", P)]:
    lp = sum(torch.log(PP[stoi[a], stoi[b]]).item() for a, b in zip(".andrejq", "andrejq."))
    print(f"{label}：'andrejq' 的平均 NLL = {-lp / 8:.3f}")

不平滑时 'andrejq' 的 NLL 是无穷大——模型宣称这个名字"绝对不可能"。

### 神经 bigram：用梯度下降找到同一个答案

换个角度看同一个问题：输入 = 当前字符（one-hot，27 维），输出 = 27 个 logits，softmax 成概率，损失 = NLL。模型只有一个 $27 \times 27$ 的权重矩阵 $W$——还记得上周说的 **one-hot 乘矩阵 = 查表**吗？`one_hot(x) @ W` 就是取 $W$ 的第 x 行。梯度下降会把 $W$ 推向"对数计数"，最终逼近计数模型的解。

In [ ]:
xs, ys = [], []
for w in words:
    cs = ["."] + list(w) + ["."]
    for ch1, ch2 in zip(cs, cs[1:]):
        xs.append(stoi[ch1])
        ys.append(stoi[ch2])
xs, ys = torch.tensor(xs), torch.tensor(ys)       # 各 (228146,)：输入字符与目标字符
print("训练样本数:", xs.nelement())

In [ ]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)
xenc = F.one_hot(xs, num_classes=27).float()                  # (228146, 27)

for k in range(200):
    # 在这里写你的代码
    raise NotImplementedError
    if k % 50 == 0:
        print(f"step {k:3d}: NLL {nll_neural.item():.4f}")
print(f"神经 bigram 最终 NLL {nll_neural.item():.4f}   （计数模型 {nll:.4f}）")
ck.neural_bigram_nll(nll_neural.item())

In [ ]:
P_neural = F.softmax(W.detach(), dim=1)                     # (27, 27)
top5 = torch.argsort(P[0], descending=True)[:5].tolist()
print("名字的首字母（'.' 那一行）概率最高的 5 个：")
for i in top5:
    print(f"  {itos[i]}   计数模型 {P[0, i]:.4f}   神经模型 {P_neural[0, i]:.4f}")

两条路殊途同归。那为什么要绕远路用神经网络？**计数表不能扩展**：想看前 3 个字符，就要 $27^3 \approx 2$ 万行的表，而且大部分格子永远是 0；神经网络通过共享参数，可以把上下文拉长到几十、几百个字符——这就是 S2 的 GPT。makemore 仓库后面几讲（MLP → RNN → Transformer）走的正是这条路。

### 延伸阅读：microgpt

Karpathy 2026 年 2 月发布了 [microgpt](https://gist.github.com/karpathy/8627fe009c40f57531cb18360106ce95)：**约 200 行、零依赖的纯 Python**，从分词、标量自动微分、注意力到 Adam 优化器，完整训练并运行一个 GPT——用的正是今天的 `names.txt`。强烈推荐学完本节后通读一遍，重点看三处：

1. 它的 `Value` 类，和第 7 周你写的 micrograd **几乎一模一样**；
2. 注意力部分，对照上周的 `scaled_dot_product` 和下面 S2 的 `Head`；
3. 训练循环，对照第 8 周的五步。

一个文件把整门课的前半段缝在了一起。（该 gist 没有附许可证声明，所以课程只提供链接、不把代码复制进仓库。）

---
## S2 · 字符级 GPT：在莎士比亚全集上训练（照 gpt.py）

### 数据与字符级分词

Tiny Shakespeare：莎剧全集的一个约 111 万字符的子集（第 2 周数过它的词频）。分词就用最朴素的办法：**每个字符一个 token**。要写四样东西：字符表 `chars`、`stoi`、`itos`、以及 `encode` / `decode`：

In [ ]:
text = data.load_shakespeare()
print(f"{len(text):,} 个字符；开头是：")
print(text[:200])

In [ ]:
# 在这里写你的代码
raise NotImplementedError
print(vocab_size, "个字符:", repr("".join(chars)))
print(encode("hii there"), "→", repr(decode(encode("hii there"))))
ck.stoi_itos(stoi, itos)
ck.encode_decode(encode, decode)

字符级分词词表小（65）、永远不会遇到"没见过的词"，代价是序列很长。真实的大模型用**子词**分词（如 BPE：把常见的字符组合并成一个 token，GPT-2 的词表约 5 万），是两者的折中——第 14 周用预训练模型时会见到。

整份文本变成一长串整数，**前 90% 训练、后 10% 验证**（按位置切，不打乱——打乱会让相邻的文本同时出现在两边，等于泄漏）：

In [ ]:
data_ids = torch.tensor(encode(text), dtype=torch.long)     # (1115394,)
n = int(0.9 * len(data_ids))
train_data, val_data = data_ids[:n], data_ids[n:]
print("训练", len(train_data), "个字符；验证", len(val_data), "个字符")

# 基线阶梯的前两级：均匀瞎猜、计数 bigram（在训练段上数、在验证段上评估）
counts = torch.zeros((vocab_size, vocab_size))
counts.index_put_((train_data[:-1], train_data[1:]), torch.ones(len(train_data) - 1), accumulate=True)
P_char = (counts + 1) / (counts + 1).sum(dim=1, keepdim=True)
bigram_val = -torch.log(P_char[val_data[:-1], val_data[1:]]).mean().item()
print(f"基线：uniform {math.log(vocab_size):.3f} → bigram {bigram_val:.3f} nats（验证集）")

### get_batch：一段文本里藏着 block_size 个训练样本

每次随机截取 `batch_size` 段、每段 `block_size` 个字符作为输入 `x`；目标 `y` 是**同一段右移一位**。一段长度为 $T$ 的片段同时提供 $T$ 个训练样本：看到第 1 个字符预测第 2 个、看到前 2 个预测第 3 个……（gpt.py 里 `get_batch(split)` 直接读全局变量；我们把数据和尺寸作为参数传进来，便于检查和在作业里复用。）

In [ ]:
def get_batch(data, block_size, batch_size):
    """随机取 batch_size 段长为 block_size 的片段：x 是输入，y 是 x 右移一位"""
    # 在这里写你的代码
    raise NotImplementedError
    return x.to(device), y.to(device)


torch.manual_seed(1337)
xb, yb = get_batch(train_data, block_size=8, batch_size=4)
print("x:", tuple(xb.shape), " y:", tuple(yb.shape))
for t in range(8):
    print(f"  看到 {decode(xb[0, :t + 1].tolist())!r:12} → 预测 {decode([yb[0, t].item()])!r}")
ck.get_batch_shapes(get_batch)

### 因果掩码：只许看过去

"看到前 $t$ 个字符预测第 $t+1$ 个"——所以位置 $t$ 绝不能偷看 $t$ 之后的字符。上周的因果掩码派上用场。Karpathy 在视频里用一个小例子讲透它：把权重矩阵的未来位置填成 $-\infty$ 再 softmax，每一行就变成"对过去的位置求平均"：

In [ ]:
T = 4
tril = torch.tril(torch.ones(T, T))                                  # (T, T) 下三角
# 在这里写你的代码
raise NotImplementedError
print(wei)
ck.causal_mask(tril)

第 3 行是 $[1/3, 1/3, 1/3, 0]$：位置 3 平均了前三个位置，看不到第四个。真正的注意力里，`wei` 不是全 0，而是 $q \cdot k / \sqrt{d}$——"平均"升级为"按相似度加权平均"。

### 搭 GPT：Head → MultiHeadAttention → FeedForward → Block → GPTLanguageModel

结构照 gpt.py。和第 11 周的积木逐一对应：

| 第 11 周（UvA） | 第 12 周（gpt.py） | 区别 |
| --- | --- | --- |
| `scaled_dot_product` + mask | `Head` | 每个头自带 `key/query/value` 三个 Linear，因果掩码写死在里面 |
| `MultiheadAttention`（合并的 `qkv_proj`） | `MultiHeadAttention`（一组 `Head`） | 去掉 q/k/v 的偏置后数学等价（UvA 的 `qkv_proj` 带偏置，gpt.py 的 key/query/value 是 `bias=False`）；gpt.py 的写法更直白、稍慢 |
| `EncoderBlock`（Post-LN） | `Block`（**Pre-LN**） | `x = x + sa(ln1(x))`：先归一化再进子层，更好训 |
| 正弦 `PositionalEncoding` | `position_embedding_table` | **可学习**的位置 embedding：每个位置一个向量 |
| mean-pool + 分类头 | `lm_head` | 每个位置都输出 65 个 logits：预测下一个字符 |

我们把 gpt.py 里的全局超参数（`n_embd`、`block_size`、`dropout`）改成了构造参数，作业里换配置做消融会方便很多。

> **对照视频时注意注意力的缩放**：视频里现场写的 gpt.py 在 `Head.forward` 中是 `wei = q @ k.transpose(-2,-1) * C**-0.5`，这里 `C = n_embd`，是**所有头加起来**的维度。仓库在 2023-01-25 的提交 988848a（"scale by head size"）中把它改成了 `k.shape[-1]**-0.5`，即除以 $\sqrt{\text{head\_size}}$。本课程采用修正后的写法，与 Vaswani et al. 2017 §3.2.1 的 $\sqrt{d_k}$（$d_k$ = 每个头的维度）一致：按第 11 周的推导，每个头的点积是 head_size 项之和、方差是 head_size，该除的是 $\sqrt{\text{head\_size}}$；除以 $\sqrt{C}$ 会让 logits 偏小（本课 2 个头时小 $\sqrt{2}$ 倍）、注意力偏平。读别人的代码时对照原理，就能发现这种不报错的小差别。

In [ ]:
class Head(nn.Module):
    """一个因果自注意力头（= 第 11 周的 scaled_dot_product + 因果掩码）"""

    def __init__(self, n_embd, head_size, block_size, dropout=0.0):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))   # 不是参数，不参与训练
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        # 在这里写你的代码
        raise NotImplementedError
        return out


class MultiHeadAttention(nn.Module):
    """多个头并行，拼接后投影回 n_embd"""

    def __init__(self, n_embd, num_heads, block_size, dropout=0.0):
        super().__init__()
        head_size = n_embd // num_heads
        self.heads = nn.ModuleList([Head(n_embd, head_size, block_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)             # (B, T, n_embd)
        return self.dropout(self.proj(out))                              # (B, T, n_embd)


class FeedForward(nn.Module):
    """逐 token 的两层 MLP（中间放大 4 倍）"""

    def __init__(self, n_embd, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.ReLU(),
                                 nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Transformer 块：先交流（注意力）、再思考（FFN）；Pre-LN 残差"""

    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        self.sa = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # 在这里写你的代码
        raise NotImplementedError
        return x

In [ ]:
class GPTLanguageModel(nn.Module):
    """字符级 GPT（结构照 gpt.py）；use_pos=False 时去掉位置 embedding（作业的消融开关）"""

    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout=0.0, use_pos=True):
        super().__init__()
        self.block_size, self.use_pos = block_size, use_pos
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)   # 可学习的位置编码
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)                                   # 最后一层 LayerNorm
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):                  # gpt.py 的初始化：权重 ~ N(0, 0.02)，偏置 0
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):             # idx, targets: (B, T)
        B, T = idx.shape
        # 在这里写你的代码
        raise NotImplementedError
        if targets is None:
            return logits, None
        B, T, C = logits.shape
        loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))           # 每个位置一个分类
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        """从 idx (B, T) 出发，逐个采样 max_new_tokens 个字符"""
        for _ in range(max_new_tokens):
            # 在这里写你的代码
            raise NotImplementedError
        return idx

### CPU 配置

ng-video-lecture 的原版配置（6 层、6 头、384 维、上下文 256，约 1000 万参数）需要 GPU：作为参照，nanoGPT 的 README 报告同规模的 shakespeare-char 配置在一块 A100 上训练约 3 分钟；Colab 免费的 T4 会明显更久（本课未实测）。这里用一个缩小版——**2 层、2 头、64 维、上下文 64、batch 32、AdamW 1e-3、2000 步**，十一万参数，普通笔记本 CPU 一分钟内训完：

In [ ]:
n_layer, n_head, n_embd = 2, 2, 64
block_size, batch_size = 64, 32
learning_rate, max_iters = 1e-3, 2000
eval_interval, eval_iters = 500, 100

experiments.set_seed(1337)                                  # gpt.py 的种子
model = GPTLanguageModel(vocab_size, n_embd, n_head, n_layer, block_size).to(device)
print(f"参数量 {sum(p.numel() for p in model.parameters()):,}")

xb, yb = get_batch(train_data, block_size, batch_size)
_, loss0 = model(xb, yb)
print(f"训练前的 loss {loss0.item():.3f}（随机初始化应 ≈ ln 65 = {math.log(65):.3f}）")
start = torch.zeros((1, 1), dtype=torch.long, device=device)  # 从换行符（id 0）开始
print("训练前的生成：", repr(decode(model.generate(start, 80)[0].tolist())))

训练前的 loss 等于 $\ln 65$：随机初始化的模型对 65 个字符一视同仁——又是第 9 周"检查初始损失"的老习惯。训练循环照 gpt.py：每隔 `eval_interval` 步用 `estimate_loss` 在训练集和验证集上各抽 `eval_iters` 个 batch 估计一次 loss（单个 batch 的 loss 噪声太大）。核心仍是第 8 周的五步：

In [ ]:
@torch.no_grad()
def estimate_loss(model, train_data, val_data, block_size, batch_size=32, eval_iters=100):
    """在训练集、验证集上各取 eval_iters 个随机 batch，返回平均 loss（照 gpt.py）"""
    out = {}
    model.eval()
    for split, d in [("train", train_data), ("val", val_data)]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(d, block_size, batch_size)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


def train_gpt(model, train_data, val_data, block_size, batch_size=32, max_iters=2000,
              learning_rate=1e-3, eval_interval=500):
    """gpt.py 的训练循环；返回 [(step, train_loss, val_loss), ...]"""
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    history = []
    t0 = time.perf_counter()
    for it in range(max_iters):
        if it % eval_interval == 0 or it == max_iters - 1:
            losses = estimate_loss(model, train_data, val_data, block_size, batch_size)
            history.append((it, losses["train"], losses["val"]))
            print(f"step {it:4d}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}"
                  f"   （{time.perf_counter() - t0:.0f} 秒）")
        # 在这里写你的代码
        raise NotImplementedError
    return history


history = train_gpt(model, train_data, val_data, block_size, batch_size, max_iters, learning_rate, eval_interval)
val_loss = history[-1][2]
ck.tiny_gpt_val_loss(val_loss)

In [ ]:
steps = [h[0] for h in history]
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(steps, [h[1] for h in history], "o-", label="train")
ax.plot(steps, [h[2] for h in history], "o-", label="val")
ax.axhline(math.log(vocab_size), color="gray", ls=":", label=f"uniform ({math.log(vocab_size):.2f})")
ax.axhline(bigram_val, color="C3", ls="--", label=f"bigram ({bigram_val:.2f})")
ax.set_xlabel("Training step")
ax.set_ylabel("Cross-entropy loss (nats)")
ax.set_title("Tiny GPT beats the bigram baseline within 500 steps")
ax.legend(loc="center right")
plt.show()

基线阶梯：**uniform 4.17 → bigram 2.48 → tiny GPT ≈ 1.8**。注意 train 与 val 的差距开始拉开：再训下去就会过拟合（原版配置加了 dropout 0.2 正是为此）。

### 采样与温度

生成时对 logits 先除以**温度** $T$ 再 softmax：$p_i = \dfrac{e^{z_i / T}}{\sum_j e^{z_j / T}}$。$T < 1$ 把分布"捏尖"——更保守、更重复；$T > 1$ 把分布"摊平"——更大胆、更多错字；$T \to 0$ 等于永远取最大概率（贪心）。（gpt.py 的 `generate` 没有温度参数，我们照 nanoGPT 加上了。）

In [ ]:
z = torch.tensor([2.0, 1.0, 0.5, -1.0])
for T_ in [0.5, 1.0, 1.5]:
    print(f"T = {T_}: 概率 {F.softmax(z / T_, dim=-1).numpy().round(3)}")

for T_ in [0.5, 1.0, 1.5]:
    torch.manual_seed(0)
    print(f"\n========== temperature {T_} ==========")
    print(decode(model.generate(start, max_new_tokens=250, temperature=T_)[0].tolist()))

---
## S3 · 换一份语料：名字，以及"怎么评估一个生成模型"

同一个 `GPTLanguageModel`，一行不改，换成 S1 的名字数据。做法：把名字用 `.` 串成一条长字符流（`.emma.olivia.ava.`），每个 `.` 既是上一个名字的结束、也是下一个名字的开始。

**划分按名字、不按字符**：先把名字打乱，90% 的名字进训练流、10% 进验证流——如果直接按字符位置切，切口处会有一个名字前半截在训练、后半截在验证。

In [ ]:
names_shuffled = words.copy()
random.Random(42).shuffle(names_shuffled)
n_tr = int(0.9 * len(names_shuffled))
train_names, val_names = names_shuffled[:n_tr], names_shuffled[n_tr:]
train_stream = "." + ".".join(train_names) + "."
val_stream = "." + ".".join(val_names) + "."

name_chars = sorted(set(train_stream))                     # '.' 排在字母前面，编号 0
name_stoi = {ch: i for i, ch in enumerate(name_chars)}
name_itos = {i: ch for i, ch in enumerate(name_chars)}
name_train = torch.tensor([name_stoi[c] for c in train_stream])
name_val = torch.tensor([name_stoi[c] for c in val_stream])
print(f"{len(name_chars)} 个字符；训练 {len(train_names)} 个名字 / {len(name_train):,} 个字符；"
      f"验证 {len(val_names)} 个名字 / {len(name_val):,} 个字符")

# 同一划分上的 bigram 基线
cnt = torch.zeros((27, 27))
cnt.index_put_((name_train[:-1], name_train[1:]), torch.ones(len(name_train) - 1), accumulate=True)
P_name = (cnt + 1) / (cnt + 1).sum(dim=1, keepdim=True)
name_bigram_val = -torch.log(P_name[name_val[:-1], name_val[1:]]).mean().item()
print(f"bigram 验证 NLL {name_bigram_val:.4f}，困惑度 {math.exp(name_bigram_val):.1f}")

名字最长 15 个字母，上下文取 32 就足够覆盖"上一个名字的尾巴 + 当前名字"。其余配置与 S2 相同：

In [ ]:
experiments.set_seed(42)
name_block = 32
name_model = GPTLanguageModel(len(name_chars), n_embd, n_head, n_layer, name_block).to(device)
name_hist = train_gpt(name_model, name_train, name_val, name_block, batch_size, max_iters, learning_rate, eval_interval)
name_val_loss = name_hist[-1][2]
print(f"\n困惑度：uniform 27 → bigram {math.exp(name_bigram_val):.1f} → GPT {math.exp(name_val_loss):.1f}")

困惑度是第一个指标：在**没见过的名字**上，模型平均每一步在几个字符之间犹豫。但它只回答"模型给真实数据多高的概率"，不回答"模型生成的东西好不好"。生成 1000 多个名字，再看两个指标：

- **长度分布**：生成名字的长度分布是否像训练集？（用总变差距离 TV = ½ Σ |p − q| 量化，0 = 完全相同、1 = 完全不重叠）
- **新颖度**：生成的名字里，有多少比例**不在**训练集中？太低 = 在背训练集；但太高也未必好——随机乱码的新颖度是 100%。

In [ ]:
def sample_gpt_names(model, n_seqs=100, length=80, temperature=1.0):
    """从 '.' 出发生成 n_seqs 段字符，按 '.' 切成名字（丢掉首尾不完整的片段）"""
    start = torch.zeros((n_seqs, 1), dtype=torch.long, device=device)      # '.' 的编号是 0
    out = model.generate(start, length, temperature).tolist()
    names = []
    for row in out:
        pieces = "".join(name_itos[i] for i in row).split(".")[1:-1]
        names += [p for p in pieces if p]
    return names


def sample_bigram_names(P, n, generator):
    names = []
    while len(names) < n:
        ix, out = 0, []
        while True:
            ix = torch.multinomial(P[ix], 1, replacement=True, generator=generator).item()
            if ix == 0:
                break
            out.append(name_itos[ix])
        if out:
            names.append("".join(out))
    return names


torch.manual_seed(0)
gpt_names = sample_gpt_names(name_model)
bigram_names = sample_bigram_names(P_name, len(gpt_names), torch.Generator().manual_seed(0))
print("GPT   :", ", ".join(gpt_names[:12]))
print("bigram:", ", ".join(bigram_names[:12]))

train_set = set(train_names)
# 在这里写你的代码
raise NotImplementedError
print(f"\n共 {len(gpt_names)} 个；新颖度：GPT {gpt_novelty:.3f}   bigram {bigram_novelty:.3f}")
ck.novelty(gpt_novelty)

In [ ]:
def length_dist(names, max_len=20):
    lens = np.clip([len(nm) for nm in names], 1, max_len)
    return np.bincount(lens, minlength=max_len + 1)[1:] / len(names)    # 长度 1..max_len 的比例


ref = length_dist(train_names)
tv_gpt = 0.5 * np.abs(length_dist(gpt_names) - ref).sum()
tv_bigram = 0.5 * np.abs(length_dist(bigram_names) - ref).sum()
print(f"长度分布的总变差距离：GPT {tv_gpt:.3f}   bigram {tv_bigram:.3f}")

fig, ax = plt.subplots(figsize=(6.5, 3.4))
x_len = np.arange(1, 21)
ax.plot(x_len, ref, "k-o", ms=3, label="training names")
ax.plot(x_len, length_dist(gpt_names), "-o", ms=3, label=f"GPT (TV {tv_gpt:.2f})")
ax.plot(x_len, length_dist(bigram_names), "-o", ms=3, label=f"bigram (TV {tv_bigram:.2f})")
ax.set_xlabel("Name length (characters, 20 = 20+)")
ax.set_ylabel("Fraction of names")
ax.set_title("GPT names follow the training length distribution more closely")
ax.legend()
plt.show()

再看温度如何同时拨动"新颖度"与"质量"：

In [ ]:
for T_ in [0.5, 1.0, 1.5]:
    torch.manual_seed(0)
    nms = sample_gpt_names(name_model, temperature=T_)
    nov = np.mean([nm not in train_set for nm in nms])
    tv = 0.5 * np.abs(length_dist(nms) - ref).sum()
    print(f"T = {T_}: 新颖度 {nov:.3f}   长度 TV {tv:.3f}   例：{', '.join(nms[:6])}")

### 怎么评估一个生成模型

三个指标各管一件事，**每一个单独拿出来都能被"刷"**：

| 指标 | 回答的问题 | 怎么作弊 |
| --- | --- | --- |
| 验证集困惑度 | 模型给没见过的真实数据多高概率 | 不能直接看出生成样本的质量 |
| 分布统计（长度等） | 生成样本的统计性质像不像真数据 | 一个只会输出"长度合适的乱码"的模型也能过 |
| 新颖度 | 是在创造还是在背书 | 随机字符串新颖度 100%；温度一调就变 |

所以报告生成模型时要**同时**报多个指标，并写明采样设置（温度！）。第 15–17 周的 VAE 与扩散模型会遇到同一个问题的图像版本。

### 选做：唐诗

想挑战中文，可以试 [chinese-poetry](https://github.com/chinese-poetry/chinese-poetry)（MIT 许可，约 5.5 万首唐诗，JSON 格式、以繁体为主）。不必下载就能先想清楚一件事：**字符表从几十个变成几千个**，模型会变成什么样？

In [ ]:
for V_ in [27, 65, 5000]:
    m_ = GPTLanguageModel(V_, n_embd, n_head, n_layer, block_size)
    total = sum(p.numel() for p in m_.parameters())
    vocab_part = m_.token_embedding_table.weight.numel() + sum(p.numel() for p in m_.lm_head.parameters())
    print(f"字符表 {V_:5d}：总参数 {total:9,}，其中 embedding + 输出头 {vocab_part:9,}（{vocab_part / total:.0%}）")

字符表 5000 时，八成多的参数沉在 embedding 和输出头上，真正做"思考"的 Transformer 块反而是少数；而且每个汉字在语料里出现的次数少得多，embedding 更难学好。这就是"词表大小如何影响模型"——也是子词分词存在的理由之一。

### 研究方法微课：vectoring

斯坦福 CS197（AI 研究方法课）的一个核心习惯叫 **vectoring**：项目开始时，列出所有"如果不成立、项目就会失败"的假设，找出**风险最大**的那一个，**最先**用最便宜的实验去验证它——而不是从最有把握、最好玩的部分做起。

一个练习用的模板（填好后放进你的 Capstone 设计文档）：

| 我的 Capstone 依赖的假设 | 错了的可能性（高/中/低） | 错了的后果 | 最便宜的验证（≤ 1 天） |
| --- | --- | --- | --- |
| 数据里的标签足够可靠 | 中 | 一切结论作废 | 随机抽 50 条人工核对 |
| 输入特征里确实有信号 | 中 | 模型都打不过多数类 | 先跑逻辑回归 / 随机森林基线 |
| 数据量够训练我想用的模型 | 高 | 深度模型输给强基线 | 学习曲线：用 10%/30%/100% 数据各训一次 |

最上面那行做完之前，别急着写复杂模型——第 10、11 周你两次看到深度模型在 5000 条数据上输给 TF-IDF，那正是"数据量够不够"这个假设在起作用。

---
## S4 · Capstone 实验设计自审

把你的设计按下面 8 项写出来（对应作业 B 的设计文档模板），每项一两句话。最好再**限时 5 分钟讲一遍**——对着自己、学习伙伴或录屏都行；讲不出来的项就是你下周要补的：

| # | 内容 | 讲清楚的标志 |
| --- | --- | --- |
| 1 | **研究问题** | 一句话，可以用"是 / 否"或一个数回答 |
| 2 | **假设** | 你预期的结果，以及**什么结果会推翻它** |
| 3 | **数据与划分** | 来源、规模、是否已拿到；训练/验证/测试怎么分，如何避免泄漏 |
| 4 | **基线** | 至少一个"不聪明但诚实"的参照（多数类、逻辑回归、随机森林、计数模型……） |
| 5 | **自变量与控制变量** | 这次只改什么；哪些保持不变；哪些干扰超参数要单独调优 |
| 6 | **指标** | 用什么衡量、为什么是它；报几个种子、均值 ± 标准差 |
| 7 | **消融计划** | 去掉或替换哪个组件，来证明它真的有用 |
| 8 | **最大风险** | vectoring：最可能让项目失败的假设，以及你**第一周**就怎么验证它 |

### 审稿人问题清单

换成审稿人的视角，逐条问自己（有学习伙伴就互相问；也可以把设计文档交给 AI 助手，让它照这份清单扮演审稿人追问）：

- 你的测试集会不会以某种方式"见过"训练数据？（同一个病人 / 同一个分子家族 / 同一批次的样本分在了两边？）
- 如果你的模型赢了基线，你怎么确定不是因为它**多调了参**？
- 你的指标在类别不平衡时还可靠吗？
- 三个种子的标准差大概多大？你预期的提升比它大吗？
- 消融里去掉的那个组件，有没有可能**换一个更简单的东西**就能达到同样效果？
- 你说的最大风险，如果第一周验证下来真的成立了，Plan B 是什么？
- 这件事在 Colab 免费 GPU 上，几个小时内跑得完吗？

把自审中发现的问题记下来——作业 B 要逐条回应。

### 本周小结

- 语言模型 = 预测下一个 token 的分类器；损失 = 平均 NLL = 交叉熵；困惑度 = $e^{\text{NLL}}$
- 基线阶梯：uniform（$\ln V$）→ 计数 bigram → 神经 bigram（同一个解）→ GPT
- 字符级分词：`stoi` / `itos` / `encode` / `decode`；`get_batch` 里 y 是 x 右移一位，一段长 T 的片段含 T 个训练样本
- GPT = token embedding + 位置 embedding + N × (因果多头注意力 + FFN，Pre-LN 残差) + LayerNorm + `lm_head`
- 温度：$T<1$ 保守、$T>1$ 大胆；评估生成模型要同时报困惑度、分布统计、新颖度，并写明温度
- vectoring：最先验证最可能让项目失败的假设

**完整版在哪里**：斯坦福 CS336（从零构建语言模型）2025 春季的作业 1 [stanford-cs336/assignment1-basics](https://github.com/stanford-cs336/assignment1-basics)（MIT 许可）要求从零实现 BPE 分词器、Transformer 语言模型、交叉熵与 AdamW、训练循环和生成——本周的内容就是它的"玩具版"。研究生课的第一个作业，你已经具备读懂它的全部概念；想做"语言模型方向"的 Capstone，可以从它的 handout 里找选题灵感。